# E005 — Robust Population CEM + Policy Zoo

Search the low-dimensional macro policy, but **do not** optimize a brittle pure best response. The robust objective blends population expectation, worst-archetype value, and lower-tail CVaR.

## Kaggle input checklist
- **Required:** repository source / V2 suite
- **Recommended:** E003 `macro_library.json` and `archetype_profiles.parquet` for extending `OPPONENT_FACTORIES`
- **Accelerator:** None / CPU
- **Internet:** Off
- **Outputs:** `cem_best.json`, `policy_params.json`, `policy_matchups.parquet`
- **Promotion rule:** serious runs should use both seats, many seeds, and a replay-derived opponent zoo.


In [ ]:
from pathlib import Path
import sys,json,numpy as np,pandas as pd
candidates=[Path('/kaggle/input'),Path.cwd().parent,Path.cwd()]
repo_file=next((p for r in candidates for p in r.rglob('src/kagv2/cem.py')),None)
if repo_file is None: raise FileNotFoundError('Attach the kaggriculture repository')
ROOT=repo_file.parents[2];sys.path.insert(0,str(ROOT));sys.path.insert(0,str(ROOT/'src'))
WORK=Path('/kaggle/working/kagv2') if Path('/kaggle/working').exists() else ROOT/'artifacts';WORK.mkdir(parents=True,exist_ok=True)
from kagv2.cem import cem_optimize_population,save_search
from kagv2.simulator import Game
from submission.parametric_agent import ParametricMind,DEFAULT_PARAMS
from submission.base_controller import HarvestMind
from baselines.v1.counter_agent import CounterMeta,TournamentMind
print('ROOT',ROOT,'WORK',WORK)


In [ ]:
OPPONENT_FACTORIES={'harvest':HarvestMind,'counter_meta':CounterMeta,'tournament_v1':TournamentMind}
SEEDS=list(range(4))  # smoke test; use 16+ for a serious run

def matchup_score(params, OppFactory, seeds=SEEDS):
    score=0.;games=0
    for seed in seeds:
        for seat in (0,1):
            mine=ParametricMind(params).act;opp=OppFactory().act
            agents=[mine,opp] if seat==0 else [opp,mine]
            cash=Game(seed=10000*seed+seat).run(agents)
            a,b=(cash[0],cash[1]) if seat==0 else (cash[1],cash[0])
            score += 1. if a>b else .5 if a==b else 0.; games+=1
    return score/games

def evaluate_vector(params):
    return np.array([matchup_score(params,f)-.5 for f in OPPONENT_FACTORIES.values()])

best,hist=cem_optimize_population(evaluate_vector,iterations=3,population=16,elite_frac=.25,worst_weight=.30,cvar_weight=.20,callback=print)
print('BEST',best);save_search(WORK/'cem_best.json',best,hist)


In [ ]:
policies={'default':dict(DEFAULT_PARAMS),'cem_robust':dict(best)}
(WORK/'policy_params.json').write_text(json.dumps(policies,indent=2,sort_keys=True))
rows=[]
for pname,params in policies.items():
    for oname,Of in OPPONENT_FACTORIES.items():
        for seed in SEEDS:
            for seat in (0,1):
                mine=ParametricMind(params).act;opp=Of().act;agents=[mine,opp] if seat==0 else [opp,mine]
                cash=Game(seed=20000+10000*seed+seat).run(agents);a,b=(cash[0],cash[1]) if seat==0 else (cash[1],cash[0])
                rows.append({'policy':pname,'opponent_archetype':oname,'seed':seed,'seat':seat,'score':1. if a>b else .5 if a==b else 0.,'margin':a-b})
matchups=pd.DataFrame(rows);matchups.to_parquet(WORK/'policy_matchups.parquet',index=False)
display(matchups.groupby(['policy','opponent_archetype']).score.agg(['mean','count']))


## Serious-run settings
Use roughly 48–96 candidates per iteration, 8–15 iterations, 16–64 seeds, both seats, and replay-derived archetype opponents. The CEM search is **offline only**. E007 turns the resulting payoff table into a robust meta-equilibrium prior for the live selector.
